# 21 · No-network baselines on the corrected DMID cohort (v2) (CPU, ~5 min)
**Runtime: CPU (standard).** Run notebook 20 first. Read-only for all inputs; writes only `results/p2_dmid_v2/baselines_v2.csv` and `results/p2_dmid_v2/baselines_v2_per_image.csv`.
Same code as notebook 11 (mean training mask of each fold thresholded at 0.5; centred disc with the mean lesion area of the fold's training crops), applied to the v2 DMID crops.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, cv2
PROJECT = '/content/drive/MyDrive/quantum-placement'
CBIS = f'{PROJECT}/data/processed/cbis_mass_256_jit'
V2 = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
OUT = f'{PROJECT}/results/p2_dmid_v2'
A2 = pd.read_csv(f'{OUT}/assignments_v2.csv')
ids = A2.roi_id[A2.status_v2 == 'retained'].tolist()
missing = [r for r in ids if not os.path.exists(f'{V2}/masks/{r}.png')]
print('retained v2 records:', len(ids), '| missing masks:', len(missing)); assert not missing

In [ ]:
def load(d, ids):
    return np.stack([cv2.imread(f'{d}/masks/{r}.png', 0) > 127 for r in ids])
F = pd.read_csv(f'{CBIS}/folds.csv'); YC = load(CBIS, F.roi_id); YD = load(V2, ids)
def dice(Y, P):
    tp = (Y & P).sum((1, 2)); s = Y.sum((1, 2)) + P.sum((1, 2))
    return np.where(s == 0, 1.0, 2 * tp / np.maximum(s, 1))
def disc(area, n=256):
    r = np.sqrt(area / np.pi); yy, xx = np.mgrid[:n, :n]
    return (yy - (n - 1) / 2) ** 2 + (xx - (n - 1) / 2) ** 2 <= r ** 2
rows, per = [], []
for k in range(5):
    f = F.fold.values; te = f == k; tr = (f != k) & (f != (k + 1) % 5)
    shapes = {'mean_mask': YC[tr].mean(0) > 0.5, 'centred_disc': disc(YC[tr].sum((1, 2)).mean())}
    for nm, sh in shapes.items():
        dc = dice(YC[te], sh[None]); dd = dice(YD, sh[None])
        rows.append(dict(baseline=nm, fold=k, dataset='CBIS', dice=float(dc.mean())))
        rows.append(dict(baseline=nm, fold=k, dataset='DMID', dice=float(dd.mean())))
        per += [dict(baseline=nm, fold=k, roi_id=r, dice=float(v)) for r, v in zip(ids, dd)]
b = pd.DataFrame(rows); b.to_csv(f'{OUT}/baselines_v2.csv', index=False); pd.DataFrame(per).to_csv(f'{OUT}/baselines_v2_per_image.csv', index=False)
print(b.groupby(['baseline', 'dataset']).dice.agg(['mean', 'std']).round(4).to_string())
print('v1 values for comparison: centred disc 0.4371 (CBIS) / 0.4265 (DMID); mean mask 0.2258 / 0.2236')
print('DONE')